# 🎮 VR World Recommendation System: Complete Analysis with Data Outputs
## From Data Exploration to Model Implementation

This enhanced notebook provides a complete journey with text-friendly outputs alongside visualizations:
1. **Data Understanding**: What data do we have? What patterns exist?
2. **User Behavior Analysis**: How do users interact with VR worlds?
3. **Problem Identification**: Why do we need recommendations?
4. **Model Selection Rationale**: Why ALS and Random Forest?
5. **Model Implementation & Comparison**: Which approach works better?

**Note**: All data outputs are printed in text format for easy copying and further processing.

In [ ]:
# === Imports & Setup ===
import os, random
import numpy as np, pandas as pd
from scipy import sparse
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Try importing optional packages
try:
    import seaborn as sns
    sns.set(style="whitegrid", font_scale=1.2)
except ImportError:
    print("Warning: seaborn not installed. Using matplotlib only.")
    sns = None

try:
    import plotly.express as px
except ImportError:
    print("Warning: plotly not installed. Some visualizations will be skipped.")
    px = None

# Set visual parameters
plt.rcParams["figure.figsize"] = (8,6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 11
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Environment ready.")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Random seed: {RANDOM_STATE}")

## 📊 Step 1: Data Loading and Initial Inspection

In [ ]:
# === Load CSV data ===
print("Loading data files...")
print("="*60)

for fn in ['users.csv','vrworld.csv','user_actions.csv','time_spent.csv']:
    if not os.path.exists(fn):
        raise FileNotFoundError(f"Missing file: {fn}")
    else:
        print(f"✓ Found: {fn}")

users = pd.read_csv('users.csv', dtype=str)
worlds = pd.read_csv('vrworld.csv', dtype=str)
actions = pd.read_csv('user_actions.csv', dtype=str)
time_spent = pd.read_csv('time_spent.csv', dtype=str)

print("\n" + "="*60)
print("Dataset shapes:")
print(f"  Users:       {users.shape[0]:,} rows × {users.shape[1]} columns")
print(f"  VR Worlds:   {worlds.shape[0]:,} rows × {worlds.shape[1]} columns")
print(f"  User Actions: {actions.shape[0]:,} rows × {actions.shape[1]} columns")
print(f"  Time Spent:  {time_spent.shape[0]:,} rows × {time_spent.shape[1]} columns")

In [ ]:
# === Display sample data from each dataset ===
print("="*60)
print("SAMPLE DATA FROM EACH DATASET")
print("="*60)

print("\n📋 USERS (first 5 rows):")
print("-"*40)
print(users.head().to_string())
print(f"\nColumn types:")
for col in users.columns:
    print(f"  {col}: {users[col].dtype}, {users[col].nunique()} unique values")

print("\n🌍 VR WORLDS (first 5 rows):")
print("-"*40)
print(worlds.head().to_string())
print(f"\nColumn types:")
for col in worlds.columns:
    print(f"  {col}: {worlds[col].dtype}, {worlds[col].nunique()} unique values")

print("\n🎯 USER ACTIONS (first 5 rows):")
print("-"*40)
print(actions.head().to_string())
print(f"\nColumn types:")
for col in actions.columns:
    print(f"  {col}: {actions[col].dtype}, {actions[col].nunique()} unique values")

print("\n⏱️ TIME SPENT (first 5 rows):")
print("-"*40)
print(time_spent.head().to_string())
print(f"\nColumn types:")
for col in time_spent.columns:
    print(f"  {col}: {time_spent[col].dtype}, {time_spent[col].nunique()} unique values")

In [ ]:
# === Data Overview & Basic Statistics ===
print("="*60)
print("📈 DATASET OVERVIEW & STATISTICS")
print("="*60)

# Calculate basic statistics
unique_users = users['user_id'].nunique()
unique_worlds = worlds['world_id'].nunique()
total_interactions = len(actions)
avg_actions_per_user = total_interactions / unique_users
avg_actions_per_world = total_interactions / unique_worlds

# Calculate sparsity
total_possible = unique_users * unique_worlds
actual_pairs = actions[['user_id', 'world_id']].drop_duplicates().shape[0]
sparsity = 1 - (actual_pairs / total_possible)
coverage = 1 - sparsity

# Create summary DataFrame
summary_stats = pd.DataFrame({
    'Metric': [
        'Unique Users',
        'Unique Worlds',
        'Total Interactions',
        'Unique User-World Pairs',
        'Total Possible Pairs',
        'Avg Actions per User',
        'Avg Actions per World',
        'Matrix Sparsity',
        'Matrix Coverage'
    ],
    'Value': [
        f"{unique_users:,}",
        f"{unique_worlds:,}",
        f"{total_interactions:,}",
        f"{actual_pairs:,}",
        f"{total_possible:,}",
        f"{avg_actions_per_user:.2f}",
        f"{avg_actions_per_world:.2f}",
        f"{sparsity:.2%}",
        f"{coverage:.2%}"
    ]
})

print(summary_stats.to_string(index=False))

print("\n💡 Key Insights:")
print(f"  • The interaction matrix is {sparsity:.1%} sparse")
print(f"  • Only {coverage:.1%} of possible user-world pairs have interactions")
print(f"  • Each user interacts with ~{avg_actions_per_user:.0f} actions on average")
print(f"  • Each world receives ~{avg_actions_per_world:.0f} actions on average")

## 🎯 Step 2: User Behavior Analysis

In [ ]:
# === User Activity Distribution Analysis ===
print("="*60)
print("👥 USER ACTIVITY DISTRIBUTION")
print("="*60)

# Calculate user activity
user_activity = actions.groupby('user_id').size().reset_index(name='num_actions')
user_activity_stats = user_activity['num_actions'].describe()

print("\nUser Activity Statistics:")
print("-"*30)
print(user_activity_stats.to_string())

# Distribution buckets
activity_buckets = pd.cut(user_activity['num_actions'], 
                          bins=[0, 1, 5, 10, 20, 50, 100, float('inf')],
                          labels=['1', '2-5', '6-10', '11-20', '21-50', '51-100', '>100'])
activity_distribution = activity_buckets.value_counts().sort_index()

print("\nUsers by Activity Level:")
print("-"*30)
for bucket, count in activity_distribution.items():
    pct = count / len(user_activity) * 100
    print(f"  {bucket:>7} actions: {count:5,} users ({pct:5.1f}%)")

# Top users analysis
top_users = user_activity.nlargest(10, 'num_actions')
print("\nTop 10 Most Active Users:")
print("-"*30)
print(top_users.to_string(index=False))

In [ ]:
# === World Popularity Distribution Analysis ===
print("="*60)
print("🌍 WORLD POPULARITY DISTRIBUTION")
print("="*60)

# Calculate world popularity
world_popularity = actions.groupby('world_id').size().reset_index(name='num_visitors')
world_popularity_stats = world_popularity['num_visitors'].describe()

print("\nWorld Popularity Statistics:")
print("-"*30)
print(world_popularity_stats.to_string())

# Distribution buckets
popularity_buckets = pd.cut(world_popularity['num_visitors'],
                            bins=[0, 10, 50, 100, 200, 500, float('inf')],
                            labels=['1-10', '11-50', '51-100', '101-200', '201-500', '>500'])
popularity_distribution = popularity_buckets.value_counts().sort_index()

print("\nWorlds by Popularity Level:")
print("-"*30)
for bucket, count in popularity_distribution.items():
    pct = count / len(world_popularity) * 100
    print(f"  {bucket:>8} visitors: {count:5,} worlds ({pct:5.1f}%)")

# Top worlds analysis
top_worlds = world_popularity.nlargest(10, 'num_visitors')
print("\nTop 10 Most Popular Worlds:")
print("-"*30)
print(top_worlds.to_string(index=False))

In [ ]:
# === Power Law Analysis ===
print("="*60)
print("📊 POWER LAW & CONCENTRATION ANALYSIS")
print("="*60)

# User concentration
user_sorted = user_activity.sort_values('num_actions', ascending=False).reset_index(drop=True)
user_sorted['cumulative_actions'] = user_sorted['num_actions'].cumsum()
user_sorted['cumulative_pct'] = user_sorted['cumulative_actions'] / user_sorted['num_actions'].sum() * 100

# Find concentration points
user_20_pct = int(len(user_sorted) * 0.2)
user_50_pct = int(len(user_sorted) * 0.5)
actions_by_top20 = user_sorted.iloc[:user_20_pct]['cumulative_pct'].iloc[-1]
actions_by_top50 = user_sorted.iloc[:user_50_pct]['cumulative_pct'].iloc[-1]

print("\nUser Activity Concentration:")
print("-"*30)
print(f"  Top 20% of users generate {actions_by_top20:.1f}% of all actions")
print(f"  Top 50% of users generate {actions_by_top50:.1f}% of all actions")
print(f"  Bottom 50% of users generate {100-actions_by_top50:.1f}% of all actions")

# World concentration
world_sorted = world_popularity.sort_values('num_visitors', ascending=False).reset_index(drop=True)
world_sorted['cumulative_visitors'] = world_sorted['num_visitors'].cumsum()
world_sorted['cumulative_pct'] = world_sorted['cumulative_visitors'] / world_sorted['num_visitors'].sum() * 100

# Find concentration points
world_20_pct = int(len(world_sorted) * 0.2)
world_50_pct = int(len(world_sorted) * 0.5)
visits_to_top20 = world_sorted.iloc[:world_20_pct]['cumulative_pct'].iloc[-1]
visits_to_top50 = world_sorted.iloc[:world_50_pct]['cumulative_pct'].iloc[-1]

print("\nWorld Popularity Concentration:")
print("-"*30)
print(f"  Top 20% of worlds receive {visits_to_top20:.1f}% of all visits")
print(f"  Top 50% of worlds receive {visits_to_top50:.1f}% of all visits")
print(f"  Bottom 50% of worlds receive {100-visits_to_top50:.1f}% of all visits")

# Create concentration summary table
concentration_df = pd.DataFrame({
    'Percentile': ['Top 1%', 'Top 5%', 'Top 10%', 'Top 20%', 'Top 50%'],
    'Users Actions %': [
        user_sorted.iloc[:int(len(user_sorted)*0.01)]['cumulative_pct'].iloc[-1] if len(user_sorted) > 100 else 0,
        user_sorted.iloc[:int(len(user_sorted)*0.05)]['cumulative_pct'].iloc[-1] if len(user_sorted) > 20 else 0,
        user_sorted.iloc[:int(len(user_sorted)*0.10)]['cumulative_pct'].iloc[-1] if len(user_sorted) > 10 else 0,
        actions_by_top20,
        actions_by_top50
    ],
    'Worlds Visits %': [
        world_sorted.iloc[:int(len(world_sorted)*0.01)]['cumulative_pct'].iloc[-1] if len(world_sorted) > 100 else 0,
        world_sorted.iloc[:int(len(world_sorted)*0.05)]['cumulative_pct'].iloc[-1] if len(world_sorted) > 20 else 0,
        world_sorted.iloc[:int(len(world_sorted)*0.10)]['cumulative_pct'].iloc[-1] if len(world_sorted) > 10 else 0,
        visits_to_top20,
        visits_to_top50
    ]
})

print("\nConcentration Summary Table:")
print("-"*30)
print(concentration_df.to_string(index=False, float_format='%.1f'))

In [ ]:
# === Visualizations for User Behavior ===
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. User activity distribution
ax1 = axes[0, 0]
ax1.hist(user_activity['num_actions'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax1.set_xlabel('Number of Actions per User')
ax1.set_ylabel('Count')
ax1.set_title('📊 User Activity Distribution')
ax1.axvline(user_activity['num_actions'].median(), color='red', linestyle='--', 
            label=f'Median: {user_activity["num_actions"].median():.0f}')
ax1.legend()

# 2. World popularity distribution
ax2 = axes[0, 1]
ax2.hist(world_popularity['num_visitors'], bins=50, edgecolor='black', alpha=0.7, color='darkorange')
ax2.set_xlabel('Number of Visitors per World')
ax2.set_ylabel('Count')
ax2.set_title('🌍 World Popularity Distribution')
ax2.axvline(world_popularity['num_visitors'].median(), color='red', linestyle='--', 
            label=f'Median: {world_popularity["num_visitors"].median():.0f}')
ax2.legend()

# 3. User concentration curve
ax3 = axes[1, 0]
ax3.plot(range(len(user_sorted)), user_sorted['cumulative_pct'], linewidth=2, color='steelblue')
ax3.fill_between(range(len(user_sorted)), user_sorted['cumulative_pct'], alpha=0.3, color='steelblue')
ax3.set_xlabel('Users (ranked by activity)')
ax3.set_ylabel('Cumulative % of Actions')
ax3.set_title('👥 User Concentration Curve')
ax3.grid(True, alpha=0.3)
ax3.axhline(80, color='red', linestyle='--', alpha=0.5)

# 4. World concentration curve
ax4 = axes[1, 1]
ax4.plot(range(len(world_sorted)), world_sorted['cumulative_pct'], linewidth=2, color='darkorange')
ax4.fill_between(range(len(world_sorted)), world_sorted['cumulative_pct'], alpha=0.3, color='darkorange')
ax4.set_xlabel('Worlds (ranked by popularity)')
ax4.set_ylabel('Cumulative % of Visits')
ax4.set_title('🎮 World Concentration Curve')
ax4.grid(True, alpha=0.3)
ax4.axhline(80, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 🕐 Step 3: Temporal and Engagement Analysis

In [ ]:
# === Action Types Analysis ===
print("="*60)
print("🎯 ACTION TYPES ANALYSIS")
print("="*60)

action_counts = actions['action_type'].value_counts()
action_pcts = (action_counts / len(actions) * 100).round(2)

action_types_df = pd.DataFrame({
    'Action Type': action_counts.index,
    'Count': action_counts.values,
    'Percentage': action_pcts.values
})

print("\nAction Type Distribution:")
print("-"*30)
print(action_types_df.to_string(index=False))

# Action type by user analysis
user_action_types = actions.groupby(['user_id', 'action_type']).size().unstack(fill_value=0)
dominant_action_per_user = user_action_types.idxmax(axis=1).value_counts()

print("\nDominant Action Type per User:")
print("-"*30)
for action_type, count in dominant_action_per_user.items():
    pct = count / len(user_action_types) * 100
    print(f"  Type {action_type}: {count:,} users ({pct:.1f}%)")

In [ ]:
# === Time Spent Analysis ===
print("="*60)
print("⏱️ TIME SPENT ANALYSIS")
print("="*60)

# Convert time to numeric
time_spent['avg_time_minutes'] = time_spent['avg_time_minutes'].astype(float)
time_stats = time_spent['avg_time_minutes'].describe()

print("\nTime Spent Statistics (minutes):")
print("-"*30)
print(time_stats.to_string())

# Time buckets
time_buckets = pd.cut(time_spent['avg_time_minutes'],
                      bins=[0, 5, 15, 30, 60, 120, float('inf')],
                      labels=['0-5', '5-15', '15-30', '30-60', '60-120', '>120'])
time_distribution = time_buckets.value_counts().sort_index()

print("\nSession Duration Distribution:")
print("-"*30)
for bucket, count in time_distribution.items():
    pct = count / len(time_spent) * 100
    print(f"  {bucket:>7} minutes: {count:,} sessions ({pct:.1f}%)")

# Long sessions analysis
long_sessions = time_spent[time_spent['avg_time_minutes'] > 60]
print(f"\nLong Sessions (>60 minutes):")
print("-"*30)
print(f"  Count: {len(long_sessions):,} ({len(long_sessions)/len(time_spent)*100:.1f}%)")
print(f"  Average duration: {long_sessions['avg_time_minutes'].mean():.1f} minutes")
print(f"  Max duration: {long_sessions['avg_time_minutes'].max():.1f} minutes")

In [ ]:
# === Engagement Correlation Analysis ===
print("="*60)
print("📊 ENGAGEMENT CORRELATION ANALYSIS")
print("="*60)

# Merge time spent with actions
actions_with_time = actions.merge(
    time_spent[['user_id', 'world_id', 'avg_time_minutes']], 
    on=['user_id', 'world_id'], 
    how='left'
)
actions_with_time['avg_time_minutes'] = actions_with_time['avg_time_minutes'].astype(float)

# User-level aggregation
user_stats = actions_with_time.groupby('user_id').agg({
    'action_id': 'count',
    'avg_time_minutes': 'mean'
}).reset_index()
user_stats.columns = ['user_id', 'num_actions', 'avg_time']
user_stats = user_stats[user_stats['avg_time'].notna()]

# Calculate correlations
corr_matrix = user_stats[['num_actions', 'avg_time']].corr()

print("\nCorrelation Matrix:")
print("-"*30)
print(corr_matrix.to_string())

# User segments by engagement
user_stats['engagement_level'] = pd.qcut(user_stats['num_actions'], 
                                          q=4, 
                                          labels=['Low', 'Medium', 'High', 'Very High'])
engagement_summary = user_stats.groupby('engagement_level').agg({
    'num_actions': ['mean', 'median'],
    'avg_time': ['mean', 'median'],
    'user_id': 'count'
}).round(2)

print("\nUser Engagement Segments:")
print("-"*30)
print(engagement_summary.to_string())

## 🔧 Step 4: Data Preprocessing for Models

In [ ]:
# === Preprocessing & Building Interaction Matrix ===
print("="*60)
print("🔧 DATA PREPROCESSING")
print("="*60)

# Normalize column names
for df in [users, worlds, actions, time_spent]:
    df.columns = df.columns.str.strip().str.lower()

# Build encoded ids
user_encoder = LabelEncoder().fit(users['user_id'])
world_encoder = LabelEncoder().fit(worlds['world_id'])

actions['user_idx'] = user_encoder.transform(actions['user_id'])
actions['world_idx'] = world_encoder.transform(actions['world_id'])

print("\nLabel Encoding Summary:")
print("-"*30)
print(f"  Users encoded: {len(user_encoder.classes_):,}")
print(f"  Worlds encoded: {len(world_encoder.classes_):,}")

# Create weighted interactions
time_spent['minutes'] = time_spent['avg_time_minutes'].astype(float)
time_agg = time_spent.groupby(['user_id','world_id'])['minutes'].mean().reset_index()
time_agg['user_idx'] = user_encoder.transform(time_agg['user_id'])
time_agg['world_idx'] = world_encoder.transform(time_agg['world_id'])

# Combine actions and time
interactions = actions.groupby(['user_idx','world_idx']).size().reset_index(name='actions')
merged = interactions.merge(time_agg, on=['user_idx','world_idx'], how='left')
merged['weight'] = merged['actions'] + merged['minutes'].fillna(0)/merged['minutes'].fillna(0).max()

print("\nInteraction Weights Statistics:")
print("-"*30)
weight_stats = merged['weight'].describe()
print(weight_stats.to_string())

# Build sparse matrix
n_users, n_worlds = len(user_encoder.classes_), len(world_encoder.classes_)
interaction_matrix = sparse.csr_matrix(
    (merged['weight'], (merged['user_idx'], merged['world_idx'])), 
    shape=(n_users, n_worlds)
)

print("\nInteraction Matrix Properties:")
print("-"*30)
print(f"  Shape: {interaction_matrix.shape[0]} × {interaction_matrix.shape[1]}")
print(f"  Non-zero entries: {interaction_matrix.nnz:,}")
print(f"  Density: {interaction_matrix.nnz/(n_users*n_worlds)*100:.4f}%")
print(f"  Memory usage: {interaction_matrix.data.nbytes / 1024:.2f} KB")

In [ ]:
# === Train/Test Split ===
print("="*60)
print("🔄 TRAIN/TEST SPLIT")
print("="*60)

def train_test_split_interactions(interactions_df, seed=RANDOM_STATE):
    rng = np.random.RandomState(seed)
    test_rows, train_rows = [], []
    for uid, g in interactions_df.groupby('user_idx'):
        idxs = g.index.values
        if len(idxs) == 1:
            train_rows.append(idxs[0])
        else:
            test_idx = rng.choice(idxs)
            for ix in idxs:
                (test_rows if ix==test_idx else train_rows).append(ix)
    return interactions_df.loc[train_rows], interactions_df.loc[test_rows]

train_df, test_df = train_test_split_interactions(merged)

# Build train and test matrices
train_mat = sparse.csr_matrix(
    (train_df['weight'], (train_df['user_idx'], train_df['world_idx'])), 
    shape=(n_users, n_worlds)
)
test_mat = sparse.csr_matrix(
    (test_df['weight'], (test_df['user_idx'], test_df['world_idx'])), 
    shape=(n_users, n_worlds)
)

print("\nSplit Statistics:")
print("-"*30)
print(f"  Train interactions: {len(train_df):,} ({len(train_df)/len(merged)*100:.1f}%)")
print(f"  Test interactions: {len(test_df):,} ({len(test_df)/len(merged)*100:.1f}%)")
print(f"  Train matrix non-zeros: {train_mat.nnz:,}")
print(f"  Test matrix non-zeros: {test_mat.nnz:,}")

# Verify no overlap
train_pairs = set(zip(train_df['user_idx'], train_df['world_idx']))
test_pairs = set(zip(test_df['user_idx'], test_df['world_idx']))
overlap = train_pairs.intersection(test_pairs)
print(f"  Overlap between train and test: {len(overlap)} pairs")

## 🚀 Step 5: Model Implementation

In [ ]:
# === Evaluation Metrics ===
def precision_recall_ndcg_at_k(y_true, y_scores, k=10):
    """
    Calculate precision, recall and NDCG at k.
    y_true: binary 1D array (len = n_items)
    y_scores: predicted scores for each item (same length as y_true)
    """
    order = np.argsort(-y_scores)
    topk = order[:k]
    y_topk = y_true[topk]

    # Precision@k
    precision = y_topk.sum() / k

    # Recall@k
    recall = y_topk.sum() / max(1, y_true.sum())

    # NDCG@k
    gains = (2**y_topk - 1) / np.log2(np.arange(2, len(y_topk) + 2))
    dcg = gains.sum()
    ideal = (2**np.sort(y_true)[::-1][:k] - 1) / np.log2(np.arange(2, len(y_topk) + 2))
    idcg = ideal.sum()
    ndcg = dcg / idcg if idcg > 0 else 0.0

    return precision, recall, ndcg

print("Evaluation metrics defined.")

### 🔷 ALS - Collaborative Filtering

In [ ]:
# === ALS Model Training ===
print("="*60)
print("🔷 ALS MODEL (COLLABORATIVE FILTERING)")
print("="*60)

# Fix OpenBLAS threading issue
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

# Try to import implicit library
try:
    from implicit.als import AlternatingLeastSquares
    print("✓ ALS library imported successfully!")
    als_available = True
except ImportError:
    print("⚠️ Warning: 'implicit' library not installed.")
    print("To install: pip install implicit")
    print("Skipping ALS model training...")
    als_available = False
    als_metrics = {'auc': 0.5, 'ap': 0.5, 'prec@10': 0.0, 'recall@10': 0.0, 'ndcg@10': 0.0}

if als_available:
    def train_als(train_csr, factors=64, regularization=0.01, iterations=20, alpha=1.0):
        model = AlternatingLeastSquares(
            factors=factors, 
            regularization=regularization,
            iterations=iterations, 
            random_state=RANDOM_STATE,
            use_native=True, 
            use_cg=True, 
            calculate_training_loss=False
        )
        model.fit((train_csr*alpha).astype('double'), show_progress=False)
        return model

    def eval_als(model, train_mat, test_mat, k=10, max_users=200):
        aucs, aps, precs, recs, ndcgs = [], [], [], [], []
        users = np.arange(train_mat.shape[0])
        np.random.shuffle(users)
        for u in users[:max_users]:
            pos_items = test_mat[u].indices
            if len(pos_items)==0: continue
            scores = model.user_factors[u] @ model.item_factors.T
            labels = np.zeros_like(scores); labels[pos_items]=1
            aucs.append(roc_auc_score(labels, scores))
            aps.append(average_precision_score(labels, scores))
            p,r,n = precision_recall_ndcg_at_k(labels, scores, k=k)
            precs.append(p); recs.append(r); ndcgs.append(n)
        return {
            'auc': float(np.mean(aucs)),
            'ap': float(np.mean(aps)),
            f'prec@{k}': float(np.mean(precs)),
            f'recall@{k}': float(np.mean(recs)),
            f'ndcg@{k}': float(np.mean(ndcgs))
        }

    print("\nTraining ALS model...")
    print("-"*30)
    print("Hyperparameters:")
    print("  Factors: 64")
    print("  Regularization: 0.01")
    print("  Iterations: 20")
    print("  Alpha: 1.0")
    
    als_model = train_als(train_mat)
    
    if als_model:
        print("\n✓ ALS model trained successfully!")
        print(f"  User factors shape: {als_model.user_factors.shape}")
        print(f"  Item factors shape: {als_model.item_factors.shape}")
        
        print("\nEvaluating ALS model...")
        als_metrics = eval_als(als_model, train_mat, test_mat, k=10)
        
        print("\nALS Performance Metrics:")
        print("-"*30)
        for metric, value in als_metrics.items():
            print(f"  {metric:12}: {value:.4f}")
        
        # Example predictions
        print("\n📊 ALS Example Predictions (User 0):")
        print("-"*30)
        example_user = 0
        train_items = train_mat[example_user].indices
        test_items = test_mat[example_user].indices
        scores = als_model.user_factors[example_user] @ als_model.item_factors.T
        topk = np.argsort(-scores)[:10]
        
        print(f"  Training items: {train_items[:5].tolist()}..." if len(train_items) > 5 else f"  Training items: {train_items.tolist()}")
        print(f"  Test items: {test_items.tolist()}")
        print(f"  Top-10 recommendations: {topk.tolist()}")
        print(f"  Recommendation scores: {scores[topk].round(3).tolist()}")
else:
    als_model = None
    print("\n⚠️ ALS model skipped due to missing dependencies")

### 🌲 Random Forest - Content-Based

In [ ]:
# === Random Forest Model Training ===
print("="*60)
print("🌲 RANDOM FOREST MODEL (CONTENT-BASED)")
print("="*60)

def encode_categoricals(df, exclude=[]):
    for col in df.select_dtypes(include=['object']).columns:
        if col not in exclude:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df

print("\nPreparing data for Random Forest...")
print("-"*30)

# Encode categorical features
users_enc = encode_categoricals(users.copy(), exclude=['user_id'])
worlds_enc = encode_categoricals(worlds.copy(), exclude=['world_id'])

# Create positive samples
actions_enc = actions.copy().merge(users_enc, on='user_id').merge(worlds_enc, on='world_id')
actions_enc['label'] = 1

print(f"  Positive samples: {len(actions_enc):,}")

# Create negative samples
all_pairs = pd.MultiIndex.from_product(
    [users['user_id'], worlds['world_id']], 
    names=['user_id','world_id']
).to_frame(index=False)

neg = all_pairs.merge(actions_enc[['user_id','world_id']], on=['user_id','world_id'], how='left', indicator=True)
neg = neg[neg['_merge']=='left_only'].drop('_merge',axis=1)
neg = neg.sample(len(actions_enc), random_state=RANDOM_STATE)
neg = neg.merge(users_enc,on='user_id').merge(worlds_enc,on='world_id')
neg['label'] = 0

print(f"  Negative samples: {len(neg):,}")

# Combine positive and negative samples
supervised = pd.concat([actions_enc, neg], ignore_index=True)
X = supervised.drop(columns=['user_id','world_id','label'])
y = supervised['label']

print(f"  Total samples: {len(supervised):,}")
print(f"  Features: {X.shape[1]}")
print(f"  Class balance: {y.value_counts().to_dict()}")

# Encode and impute
X_enc = X.copy()
for col in X_enc.columns:
    if X_enc[col].dtype == 'object':
        le = LabelEncoder()
        X_enc[col] = le.fit_transform(X_enc[col].astype(str))

imputer = SimpleImputer(strategy="mean")
X_enc = pd.DataFrame(imputer.fit_transform(X_enc), columns=X_enc.columns)

print("\nFeature Overview:")
print("-"*30)
for col in X_enc.columns[:10]:  # Show first 10 features
    print(f"  {col}: min={X_enc[col].min():.2f}, max={X_enc[col].max():.2f}, mean={X_enc[col].mean():.2f}")

In [ ]:
# === Random Forest Training and Evaluation ===
print("\nTraining Random Forest model...")
print("-"*30)
print("Hyperparameters:")
print("  n_estimators: 100")
print("  max_depth: None (unlimited)")
print("  random_state: 42")

rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# Cross-validation
aucs, aps, precs, recs, ndcgs = [],[],[],[],[]
fold_results = []

for fold, (tr, va) in enumerate(skf.split(X, y), 1):
    print(f"  Training fold {fold}/3...")
    rf.fit(X_enc.iloc[tr], y.iloc[tr])
    prob = rf.predict_proba(X_enc.iloc[va])[:,1]
    
    auc = roc_auc_score(y.iloc[va], prob)
    ap = average_precision_score(y.iloc[va], prob)
    p, r, n = precision_recall_ndcg_at_k(y.iloc[va].values, prob, k=10)
    
    aucs.append(auc)
    aps.append(ap)
    precs.append(p)
    recs.append(r)
    ndcgs.append(n)
    
    fold_results.append({
        'Fold': fold,
        'AUC': auc,
        'AP': ap,
        'Prec@10': p,
        'Recall@10': r,
        'NDCG@10': n
    })

rf_metrics = {
    'auc': float(np.mean(aucs)),
    'ap': float(np.mean(aps)),
    'prec@10': float(np.mean(precs)),
    'recall@10': float(np.mean(recs)),
    'ndcg@10': float(np.mean(ndcgs))
}

print("\n✓ Random Forest model trained successfully!")

# Display fold results
fold_df = pd.DataFrame(fold_results)
print("\nCross-Validation Results by Fold:")
print("-"*30)
print(fold_df.to_string(index=False, float_format='%.4f'))

print("\nRandom Forest Average Performance:")
print("-"*30)
for metric, value in rf_metrics.items():
    print(f"  {metric:12}: {value:.4f}")

In [ ]:
# === Feature Importance Analysis ===
print("\n📊 Feature Importance Analysis:")
print("-"*30)

importances = rf.feature_importances_
feat_names = X_enc.columns.tolist()
fi_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
fi_df = fi_df.sort_values("Importance", ascending=False)

print("\nTop 15 Most Important Features:")
print(fi_df.head(15).to_string(index=False, float_format='%.4f'))

# Feature importance by category
user_features = [f for f in feat_names if f in users.columns]
world_features = [f for f in feat_names if f in worlds.columns]
other_features = [f for f in feat_names if f not in user_features and f not in world_features]

user_importance = fi_df[fi_df['Feature'].isin(user_features)]['Importance'].sum()
world_importance = fi_df[fi_df['Feature'].isin(world_features)]['Importance'].sum()
other_importance = fi_df[fi_df['Feature'].isin(other_features)]['Importance'].sum()

print("\nImportance by Feature Category:")
print("-"*30)
print(f"  User features:  {user_importance:.4f} ({user_importance*100:.1f}%)")
print(f"  World features: {world_importance:.4f} ({world_importance*100:.1f}%)")
print(f"  Other features: {other_importance:.4f} ({other_importance*100:.1f}%)")

## 🏆 Step 6: Model Comparison and Insights

In [ ]:
# === Comprehensive Model Comparison ===
print("="*60)
print("🏆 MODEL COMPARISON")
print("="*60)

# Create comparison table
comparison = pd.DataFrame([
    {
        'Model': 'ALS',
        'Type': 'Collaborative',
        'ROC AUC': als_metrics['auc'],
        'Avg Precision': als_metrics['ap'],
        'Precision@10': als_metrics['prec@10'],
        'Recall@10': als_metrics['recall@10'],
        'NDCG@10': als_metrics['ndcg@10']
    },
    {
        'Model': 'Random Forest',
        'Type': 'Content-Based',
        'ROC AUC': rf_metrics['auc'],
        'Avg Precision': rf_metrics['ap'],
        'Precision@10': rf_metrics['prec@10'],
        'Recall@10': rf_metrics['recall@10'],
        'NDCG@10': rf_metrics['ndcg@10']
    }
])

print("\nPerformance Metrics Comparison:")
print("-"*50)
print(comparison.to_string(index=False, float_format='%.4f'))

# Calculate relative improvements
print("\nRelative Performance (RF vs ALS):")
print("-"*50)
for col in ['ROC AUC', 'Avg Precision', 'Precision@10', 'Recall@10', 'NDCG@10']:
    als_val = comparison[comparison['Model'] == 'ALS'][col].values[0]
    rf_val = comparison[comparison['Model'] == 'Random Forest'][col].values[0]
    if als_val > 0:
        improvement = (rf_val - als_val) / als_val * 100
        print(f"  {col:15}: {improvement:+.1f}%")

In [ ]:
# === Model Characteristics Comparison ===
print("\n📋 MODEL CHARACTERISTICS COMPARISON")
print("="*60)

characteristics = pd.DataFrame({
    'Aspect': [
        'Sparsity Handling',
        'Cold Start Capability',
        'Feature Richness',
        'Interpretability',
        'Scalability',
        'Training Time',
        'Prediction Time',
        'Memory Usage'
    ],
    'ALS': [
        'Excellent - designed for sparse data',
        'Poor - needs interaction history',
        'Limited - only uses interactions',
        'Medium - latent factors',
        'Excellent - matrix operations',
        'Fast - efficient algorithms',
        'Very Fast - dot products',
        'Low - sparse matrices'
    ],
    'Random Forest': [
        'Requires negative sampling',
        'Good - uses content features',
        'Excellent - all features',
        'High - feature importance',
        'Medium - tree ensemble',
        'Slower - multiple trees',
        'Medium - tree traversal',
        'Higher - stores trees'
    ]
})

print("\nModel Characteristics:")
print("-"*60)
for _, row in characteristics.iterrows():
    print(f"\n{row['Aspect']}:")
    print(f"  ALS: {row['ALS']}")
    print(f"  RF:  {row['Random Forest']}")

In [ ]:
# === Use Case Recommendations ===
print("\n🎯 RECOMMENDATION STRATEGY")
print("="*60)

recommendations = {
    'When to use ALS': [
        'Large user base with rich interaction history',
        'Need to discover hidden patterns in user behavior',
        'Real-time recommendations required',
        'Limited content features available',
        'Working with implicit feedback data'
    ],
    'When to use Random Forest': [
        'Many new users or items (cold start)',
        'Rich feature sets available',
        'Need interpretable results',
        'Smaller datasets',
        'Explicit feedback available'
    ],
    'Hybrid Approach Benefits': [
        'Use RF for users with <5 interactions',
        'Use ALS for users with rich history',
        'Weighted ensemble based on confidence',
        'Fallback to popular items when both fail',
        'A/B test to find optimal weights'
    ]
}

for strategy, points in recommendations.items():
    print(f"\n{strategy}:")
    print("-"*40)
    for point in points:
        print(f"  • {point}")

In [ ]:
# === Visualizations for Model Comparison ===
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. ROC AUC Comparison
ax1 = axes[0, 0]
models = ['ALS', 'Random Forest']
aucs = [als_metrics['auc'], rf_metrics['auc']]
bars1 = ax1.bar(models, aucs, color=['steelblue', 'darkorange'], alpha=0.7, edgecolor='black')
ax1.set_ylabel('Score')
ax1.set_title('📊 ROC AUC Score')
ax1.set_ylim(0, 1.1)
for i, auc in enumerate(aucs):
    ax1.text(i, auc + 0.02, f'{auc:.3f}', ha='center', fontweight='bold')

# 2. Average Precision Comparison
ax2 = axes[0, 1]
aps = [als_metrics['ap'], rf_metrics['ap']]
bars2 = ax2.bar(models, aps, color=['steelblue', 'darkorange'], alpha=0.7, edgecolor='black')
ax2.set_ylabel('Score')
ax2.set_title('🎯 Average Precision')
ax2.set_ylim(0, 1.1)
for i, ap in enumerate(aps):
    ax2.text(i, ap + 0.02, f'{ap:.3f}', ha='center', fontweight='bold')

# 3. Ranking Metrics
ax3 = axes[0, 2]
x = np.arange(3)
width = 0.35
metrics_names = ['Precision@10', 'Recall@10', 'NDCG@10']
als_vals = [als_metrics['prec@10'], als_metrics['recall@10'], als_metrics['ndcg@10']]
rf_vals = [rf_metrics['prec@10'], rf_metrics['recall@10'], rf_metrics['ndcg@10']]

bars3 = ax3.bar(x - width/2, als_vals, width, label='ALS', color='steelblue', alpha=0.7)
bars4 = ax3.bar(x + width/2, rf_vals, width, label='Random Forest', color='darkorange', alpha=0.7)
ax3.set_ylabel('Score')
ax3.set_title('🏅 Ranking Metrics @10')
ax3.set_xticks(x)
ax3.set_xticklabels(metrics_names, rotation=45, ha='right')
ax3.legend()
ax3.set_ylim(0, 1.1)

# 4. Radar Chart
ax4 = axes[1, 0]
ax4.axis('off')
ax4_polar = fig.add_subplot(2, 3, 4, projection='polar')

categories = ['ROC AUC', 'Avg Precision', 'Precision@10', 'Recall@10', 'NDCG@10']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

# ALS scores
als_scores = [als_metrics['auc'], als_metrics['ap'], als_metrics['prec@10'], 
              als_metrics['recall@10'], als_metrics['ndcg@10']]
als_scores += als_scores[:1]

# RF scores
rf_scores = [rf_metrics['auc'], rf_metrics['ap'], rf_metrics['prec@10'], 
             rf_metrics['recall@10'], rf_metrics['ndcg@10']]
rf_scores += rf_scores[:1]

ax4_polar.plot(angles, als_scores, 'o-', linewidth=2, color='steelblue', label='ALS')
ax4_polar.fill(angles, als_scores, alpha=0.25, color='steelblue')
ax4_polar.plot(angles, rf_scores, 'o-', linewidth=2, color='darkorange', label='Random Forest')
ax4_polar.fill(angles, rf_scores, alpha=0.25, color='darkorange')

ax4_polar.set_xticks(angles[:-1])
ax4_polar.set_xticklabels(categories)
ax4_polar.set_ylim(0, 1)
ax4_polar.set_title('🎯 Performance Radar', y=1.08)
ax4_polar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# 5 & 6: Hide for cleaner layout
axes[1, 1].axis('off')
axes[1, 2].axis('off')

plt.suptitle('📊 Comprehensive Model Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🎓 Conclusions & Next Steps

In [ ]:
# === Final Summary ===
print("="*60)
print("🎓 FINAL SUMMARY & RECOMMENDATIONS")
print("="*60)

print("\n📊 Key Findings:")
print("-"*40)
print(f"1. Data Sparsity: {sparsity:.1%} of possible interactions are missing")
print(f"2. User Distribution: Top 20% of users generate {actions_by_top20:.1f}% of actions")
print(f"3. World Distribution: Top 20% of worlds receive {visits_to_top20:.1f}% of visits")
print(f"4. Cold Start: {(user_activity['num_actions'] <= 2).sum()} users have ≤2 interactions")
print(f"5. Model Performance: RF shows {(rf_metrics['auc']/als_metrics['auc']-1)*100:.1f}% better AUC than ALS")

print("\n💡 Insights:")
print("-"*40)
print("• Random Forest achieves near-perfect metrics (possible overfitting)")
print("• ALS struggles with extreme sparsity but provides reasonable rankings")
print("• Different metrics tell different stories - choose based on business goals")
print("• Cold start is a major challenge requiring hybrid approaches")

print("\n🚀 Recommended Next Steps:")
print("-"*40)
next_steps = [
    "Implement hybrid model: RF for cold start, ALS for warm users",
    "Collect more implicit signals: views, searches, hover time",
    "Add social features: friend recommendations, collaborative filtering",
    "Deploy A/B testing framework for continuous improvement",
    "Consider deep learning: Neural CF, Graph Neural Networks",
    "Implement real-time feature updates and model retraining",
    "Add diversity and novelty to recommendations",
    "Build fallback strategies for edge cases"
]

for i, step in enumerate(next_steps, 1):
    print(f"{i}. {step}")

print("\n🎯 Final Recommendation:")
print("-"*40)
print("For this VR world recommendation system, implement a hybrid approach:")
print("• Use Random Forest for users with <5 interactions (cold start)")
print("• Use ALS for users with rich interaction history")
print("• Weight predictions based on user activity level")
print("• Continuously monitor and retrain based on user feedback")
print("\nRemember: The best recommendation system is one that continuously")
print("learns and adapts. Start simple, measure everything, and iterate!")